# 6. Agents and Tools

Let the model dynamically choose lookup and calculation tools.

[🔊 Open Interview & Audio Practice](https://htmlpreview.github.io/?https://github.com/blaire101/langchain-course-companion-26/blob/main/docs/06_agents.html)

> GitHub notebook previews do not execute custom JavaScript. Interactive pronunciation and interview notes are provided in the companion HTML page.


## 1. Import agent components

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from dotenv import load_dotenv

load_dotenv()
MODEL = "openai:gpt-4o-mini"

## 2. Define the tools

In [ ]:
@tool
def plan_lookup(plan: str) -> str:
    """Look up known features and monthly price for a subscription plan."""
    data = {
        "starter": {"price": 29, "features": "five users"},
        "business": {"price": 99, "features": "audit logs, API access"},
    }
    return str(data.get(plan.lower(), "Unknown plan"))


@tool
def total_cost(monthly_price: float, months: int) -> float:
    """Calculate total subscription cost."""
    return monthly_price * months

## 3. Create the agent

In [ ]:
agent = create_agent(
    model=init_chat_model(MODEL, temperature=0),
    tools=[plan_lookup, total_cost],
    system_prompt="Use tools for plan facts and calculations.",
)

## 4. Invoke the agent

In [ ]:
question = (
    "What does the Business plan include "
    "and what is the cost for 12 months?"
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)
print(result["messages"][-1].content)

## 5. Inspect the tool-calling trace

In [ ]:
for index, message in enumerate(result["messages"], start=1):
    print(f"\n--- Message {index}: {type(message).__name__} ---")
    print(message)

## Example Output

```text
The Business plan includes audit logs and API access. At $99 per month, the total cost for 12 months is $1,188.
```

The exact wording may vary for model-generated responses, while the expected facts and structure should remain consistent.


## Relationship Diagram

![Agents and Tools flow](../assets/06_agents_flow.png)

### Relationship Summary

- **1. User Question**
- **2. LLM Agent**
- **3. plan_lookup**
- **4. Tool Result**
- **5. total_cost**
- **6. Tool Result**
- **7. Final Answer**

## Think Summary

### How do tool names, type hints, and docstrings affect tool selection?

- A descriptive name helps the model infer the tool purpose.
- Type hints define the argument schema.
- The docstring explains when the tool should be used.
- A strong tool has a narrow responsibility and predictable output.

### Key takeaway

A strong tool has a narrow responsibility and predictable output.
